# Full Pipeline

End-to-end: digital clock → time recognition → analog clock manipulation.

In [ ]:
import sys, torch
from IPython.display import display, Image as IPImage
from analog_clock.pipeline_utils import generate_sketch_cgan_gif, generate_inpainting_gif
from analog_clock.analog_sketch_creator import draw_analog_clock
import pipeline_runners as pr

device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

# ── paths ──────────────────────────────────────────────────────────────────
DIGITAL_CLOCK_IMAGE  = "test_digital_clock.png"
ANALOG_CLOCK_IMAGE   = "test_analog_clock.jpg"

YOLO_HH_MM_PATH      = "digital_clock/yolo_detect_hh_mm/yolo.pt"
SVHN_CNN_PATH        = "digital_clock/svhn_digit_recognition_cnn/svhn_cnn.pth"

YOLO_CLOCK_PATH      = "analog_clock/yolo_detect_clock/analog_clock_yolo_model.pt"
SKETCH_GAN_PATH      = "analog_clock/GAN/sketch/generator_100.pth"

YOLO_HANDS_PATH      = "analog_clock/yolo_detetct_hands/yolo_hands_seconds.pt"
TIME_CNN_PATH        = "analog_clock/time_recognition_cnn/clock_hand_cnn_best.pth"
INPAINTING_GAN_PATH  = "analog_clock/GAN/inpainting/inpaint_gen_100.pth"

SKETCH_GIF_PATH      = "sketch_cgan_animation.gif"
INPAINTING_GIF_PATH  = "inpainting_animation.gif"

## Stage 1 — Digital Clock Recognition

YOLO detects HH / MM bounding boxes; SVHN CNN reads each digit.

In [ ]:
yolo_digit, cnn_model = pr.load_digital_models(YOLO_HH_MM_PATH, SVHN_CNN_PATH, device)
hh_val, mm_val, sketch = pr.process_clock_image(DIGITAL_CLOCK_IMAGE, yolo_digit, cnn_model, device)

## Stage 2 — Analog Clock · Sketch-cGAN

YOLO crops the clock face; a sketch of the target time guides the Pix2Pix generator.

In [ ]:
yolo_clock, generator = pr.load_sketch_cgan_models(YOLO_CLOCK_PATH, SKETCH_GAN_PATH, device)
v1_result, v1_original, v1_crop_coords = pr.run_sketch_cgan(
    ANALOG_CLOCK_IMAGE, hh_val, mm_val, yolo_clock, generator, device
)

### Sketch-cGAN — Animated Transition

Animate the clock hands from the recognised original time to the target time.

In [ ]:
orig_hh, orig_mm = pr.recognize_original_time(ANALOG_CLOCK_IMAGE, YOLO_HANDS_PATH, TIME_CNN_PATH, device)

gif_v1 = generate_sketch_cgan_gif(
    original_img_pil   = v1_original,
    crop_coords        = v1_crop_coords,
    generator          = generator,
    transform          = pr._CGAN_TRANSFORM,
    start_hh           = orig_hh,
    start_mm           = orig_mm,
    target_hh          = hh_val,
    target_mm          = mm_val,
    draw_analog_clock_fn = draw_analog_clock,
    device             = device,
    output_path        = SKETCH_GIF_PATH,
)
print(f"Animation: {orig_hh}:{orig_mm:02d} → {hh_val}:{mm_val:02d}")
display(IPImage(filename=gif_v1))

## Stage 3 — Analog Clock · Inpainting

YOLO-Seg masks the hands; the GAN fills in the background; hands are recomposed at the target time.

In [ ]:
yolo_hands, igan = pr.load_inpainting_models(YOLO_HANDS_PATH, INPAINTING_GAN_PATH, device)
v2 = pr.run_inpainting(ANALOG_CLOCK_IMAGE, hh_val, mm_val, igan, yolo_hands, device)

### Inpainting — Animated Transition

In [ ]:
gif_v2 = generate_inpainting_gif(
    img_cv           = v2["img_cv"],
    clean_bg         = v2["clean_bg"],
    center           = v2["center"],
    mask_h           = v2["best_mask_h"],
    mask_m           = v2["best_mask_m"],
    current_angle_h  = v2["angle_h_orig"],
    current_angle_m  = v2["angle_m_orig"],
    start_hh         = orig_hh,
    start_mm         = orig_mm,
    target_hh        = hh_val,
    target_mm        = mm_val,
    output_path      = INPAINTING_GIF_PATH,
)
print(f"Animation: {orig_hh}:{orig_mm:02d} → {hh_val}:{mm_val:02d}")
display(IPImage(filename=gif_v2))